# relu-elementwise-max — ex2: in-place relu_ and verify storage identity

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `relu-elementwise-max`. Running the final beacon cell reports progress against the `CNN: ReLU as elementwise max` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ReLU as elementwise max` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`relu-elementwise-max`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "relu-elementwise-max"
DD_SUBTOPIC = "CNN: ReLU as elementwise max"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ReLU as elementwise max — in-place edition

Three forward forms compute the same values; their storage semantics differ:

```python
y = t.maximum(x, t.tensor(0.0))   # OUT-OF-PLACE — y is a NEW tensor
y = F.relu(x)                      # OUT-OF-PLACE — same
x.relu_()                          # IN-PLACE — mutates x; returns x
```

The previous drill (ex1) used `t.maximum` and inspected the sub-gradient at `x=0`. This drill targets the **in-place form `x.relu_()`** — proving via `data_ptr()` that the input storage is reused, and understanding why that saves memory but breaks autograd for non-leaf tensors that need their input for backward.

### Exercise 2 — in-place relu_ and verify storage identity

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the in-place ReLU operator `Tensor.relu_()` by applying it to a tensor and verifying via `data_ptr()` that the returned tensor shares storage with the input.
> Keywords: relu, in-place, data_ptr, storage
> ```

**KCs targeted:** `relu-max-with-zero`, `relu-in-place-aliases-input`

Implement `ex2_relu_inplace(x)` that applies ReLU **in place** and returns `(out, same_storage)` where `same_storage` is a bool: `True` iff the returned tensor's `data_ptr()` equals the input's BEFORE-call `data_ptr()`.

**Required call.** Use `x.relu_()` (the trailing underscore is PyTorch's convention for in-place ops). NOT `F.relu(x)`, NOT `t.maximum(x, 0)` — those allocate new tensors and would yield `same_storage = False`.

**Required no_grad context.** Wrap the in-place call in `with t.no_grad():` so the in-place mutation doesn't pollute the autograd graph (this is also what PyTorch's own activation modules do under the `inplace=True` flag).

Input: `x` — float tensor of any shape (NOT necessarily a leaf with `requires_grad`).
Output: `(out, same_storage)` — `out is x` and `same_storage == True`.

After return, `x` itself has had negatives replaced with 0.

In [ ]:
def ex2_relu_inplace(x: Tensor):
    """Apply ReLU in place via x.relu_(); return (out, same_storage)."""
    raise NotImplementedError()


def _test_ex2():
    x = t.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
    ptr_before = x.data_ptr()
    out, same_storage = ex2_relu_inplace(x)

    # Identity and aliasing.
    assert out is x, 'in-place op must return the SAME object'
    assert out.data_ptr() == ptr_before, 'data_ptr changed → storage was not reused'
    assert same_storage is True, f'same_storage must be True, got {same_storage!r}'

    # Forward values.
    expected = t.tensor([0.0, 0.0, 0.0, 0.5, 2.0])
    assert t.equal(x, expected), f'forward wrong: {x} vs {expected}'

    # Larger random shape — confirm storage is reused for any shape.
    rng = t.Generator().manual_seed(0)
    y = t.randn(4, 5, generator=rng)
    y_ref_relu = y.clamp(min=0).clone()
    ptr_y = y.data_ptr()
    out_y, ok_y = ex2_relu_inplace(y)
    assert ok_y is True
    assert out_y.data_ptr() == ptr_y
    assert t.allclose(y, y_ref_relu), f'larger-tensor forward wrong'

    # Anti-cheat: ensure they used .relu_(), not e.g. clamp_().
    # We can't introspect the call directly, but we can verify the mutation
    # matches relu (not, say, abs) for negatives.
    z = t.tensor([-3.0, -1.0, 4.0])
    out_z, _ = ex2_relu_inplace(z)
    assert t.equal(z, t.tensor([0.0, 0.0, 4.0])), 'mutation must match relu semantics'
    # After in-place ReLU, all entries are non-negative.
    assert (z >= 0).all().item(), 'all entries must be >= 0 post-ReLU'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_relu_inplace(x: Tensor):
    ptr_before = x.data_ptr()
    with t.no_grad():
        out = x.relu_()
    same_storage = (out.data_ptr() == ptr_before)
    return out, same_storage
```

**The underscore convention.** PyTorch suffixes in-place ops with `_`. `relu_`, `add_`, `mul_`, `clamp_` all mutate `self` and return `self`. The non-underscore counterpart (`relu`, `add`, etc.) returns a fresh tensor — `data_ptr()` differs.

**When the memory savings are worth it.** Activation modules expose `inplace=True` (e.g. `nn.ReLU(inplace=True)`) precisely because for deep networks the activation tensors are large and the gradient w.r.t. the pre-activation can be recovered from the post-activation alone (zero stays zero, positive stays positive). For most other ops, in-place saves memory but breaks autograd because the original tensor is needed for backward.

**Difference from ex1.** ex1 used the **out-of-place** `t.maximum` and verified the autograd sub-gradient at `x=0`. ex2 uses the **in-place** `relu_` and verifies storage identity via `data_ptr()`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()